In [1]:
model = "google_genai:gemini-2.5-flash"
emb_model = "google_genai:gemini-embedding-001"

## Semantic Memory as Collection

In [2]:
from langmem import create_memory_manager

manager = create_memory_manager(
    model,
    instructions="Extract all noteworthy facts, events, and relationships. Indicate their importance.",
    enable_inserts=True,
)

# Process a conversation to extract semantic memories
conversation = [
    {"role": "user", "content": "I work at Acme Corp in the ML team"},
    {"role": "assistant", "content": "I'll remember that. What kind of ML work do you do?"},
    {"role": "user", "content": "Mostly NLP and large language models"}
]

In [3]:
memories = manager.invoke({"messages": conversation})
# Example memories:
# [
#     ExtractedMemory(
#         id="27e96a9d-8e53-4031-865e-5ec50c1f7ad5",
#         content=Memory(
#             content="[IMPORTANT] User prefers to be called Lex (short for Alex) and appreciates"
#             " casual, witty communication style with relevant emojis."
#         ),
#     ),
#     ExtractedMemory(
#         id="e2f6b646-cdf1-4be1-bb40-0fd91d25d00f",
#         content=Memory(
#             content="[BACKGROUND] Lex is proficient in Python programming and specializes in developing"
#             " AI systems with a focus on making them sound more natural and less corporate."
#         ),
#     ),
#     ExtractedMemory(
#         id="c1e03ebb-a393-4e8d-8eb7-b928d8bed510",
#         content=Memory(
#             content="[HOBBY] Lex is a competitive speedcuber (someone who solves Rubik's cubes competitively),"
#             " showing an interest in both technical and recreational puzzle-solving."
#         ),
#     ),
#     ExtractedMemory(
#         id="ee7fc6e4-0118-425f-8704-6b3145881ff7",
#         content=Memory(
#             content="[PERSONALITY] Based on communication style and interests, Lex appears to value authenticity,"
#             " creativity, and technical excellence while maintaining a fun, approachable demeanor."
#         ),
#     ),
# ]

In [4]:
memories

[ExtractedMemory(id='e3c8e891-6441-49a5-936f-95b3c2bf02e6', content=Memory(content='The user works at Acme Corp.'))]

## Semantic Memory as Profiles

In [5]:
from langmem import create_memory_manager
from pydantic import BaseModel


class UserProfile(BaseModel):
    """Save the user's preferences."""
    name: str
    preferred_name: str
    response_style_preference: str
    special_skills: list[str]
    other_preferences: list[str]


manager = create_memory_manager(
    model,
    schemas=[UserProfile],
    instructions="Extract user preferences and settings",
    enable_inserts=False,
)

# Extract user preferences from a conversation
conversation = [
    {"role": "user", "content": "Hi! I'm Alex but please call me Lex. I'm a wizard at Python and love making AI systems that don't sound like boring corporate robots 🤖"},
    {"role": "assistant", "content": "Nice to meet you, Lex! Love the anti-corporate-robot stance. How would you like me to communicate with you?"},
    {"role": "user", "content": "Keep it casual and witty - and maybe throw in some relevant emojis when it feels right ✨ Also, besides AI, I do competitive speedcubing!"},
]

In [6]:
profile = manager.invoke({"messages": conversation})[0]
print(profile)
# Example profile:
# ExtractedMemory(
#     id="6f555d97-387e-4af6-a23f-a66b4e809b0e",
#     content=UserProfile(
#         name="Alex",
#         preferred_name="Lex",
#         response_style_preference="casual and witty with appropriate emojis",
#         special_skills=[
#             "Python programming",
#             "AI development",
#             "competitive speedcubing",
#         ],
#         other_preferences=[
#             "prefers informal communication",
#             "dislikes corporate-style interactions",
#         ],
#     ),
# )

ExtractedMemory(id='743e7b9a-75cf-46ef-8552-a84f23b64c89', content=UserProfile(name='Alex', preferred_name='Lex', response_style_preference='casual and witty, with relevant emojis', special_skills=['Python programming', 'AI system development'], other_preferences=['competitive speedcubing']))


## Semantic Memory as Triples

In [7]:
from pydantic import BaseModel, Field
from langmem import create_memory_manager


class Triple(BaseModel): 
    """Store all new facts, preferences, and relationships as triples."""
    subject: str
    predicate: str
    object: str
    context: str | None = None

# Configure extraction
manager = create_memory_manager(  
    model,
    schemas=[Triple], 
    instructions="Extract user preferences and any other useful information",
    enable_inserts=True,
    enable_deletes=True,
)

# First conversation - extract triples
conversation1 = [
    {"role": "user", "content": "Alice manages the ML team and mentors Bob, who is also on the team."}
]

memories = manager.invoke({"messages": conversation1})
print("After first conversation:")
for m in memories:
    print(m)
# ExtractedMemory(id='f1bf258c-281b-4fda-b949-0c1930344d59', content=Triple(subject='Alice', predicate='manages', object='ML_team', context=None))
# ExtractedMemory(id='0214f151-b0c5-40c4-b621-db36b845956c', content=Triple(subject='Alice', predicate='mentors', object='Bob', context=None))
# ExtractedMemory(id='258dbf2d-e4ac-47ac-8ffe-35c70a3fe7fc', content=Triple(subject='Bob', predicate='is_member_of', object='ML_team', context=None))

After first conversation:
ExtractedMemory(id='1d6c0292-a16c-468e-91c8-9ed06e742c36', content=Triple(subject='Alice', predicate='manages', object='ML team', context=None))
ExtractedMemory(id='cd15b8f4-29ea-44bb-be1f-c11adaa8ae2f', content=Triple(subject='Alice', predicate='mentors', object='Bob', context=None))
ExtractedMemory(id='3dbecb1f-fde0-41d9-95ca-914eb639cc1a', content=Triple(subject='Bob', predicate='is on', object='ML team', context=None))


In [8]:
# Second conversation - update and add triples
conversation2 = [
    {"role": "user", "content": "Bob now leads the ML team and the NLP project."}
]
update = manager.invoke({"messages": conversation2, "existing": memories})
print("After second conversation:")
for m in update:
    print(m)
# ExtractedMemory(id='65fd9b68-77a7-4ea7-ae55-66e1dd603046', content=RemoveDoc(json_doc_id='f1bf258c-281b-4fda-b949-0c1930344d59'))
# ExtractedMemory(id='7f8be100-5687-4410-b82a-fa1cc8d304c0', content=Triple(subject='Bob', predicate='leads', object='ML_team', context=None))
# ExtractedMemory(id='f4c09154-2557-4e68-8145-8ccd8afd6798', content=Triple(subject='Bob', predicate='leads', object='NLP_project', context=None))
# ExtractedMemory(id='f1bf258c-281b-4fda-b949-0c1930344d59', content=Triple(subject='Alice', predicate='manages', object='ML_team', context=None))
# ExtractedMemory(id='0214f151-b0c5-40c4-b621-db36b845956c', content=Triple(subject='Alice', predicate='mentors', object='Bob', context=None))
# ExtractedMemory(id='258dbf2d-e4ac-47ac-8ffe-35c70a3fe7fc', content=Triple(subject='Bob', predicate='is_member_of', object='ML_team', context=None))
existing = [m for m in update if isinstance(m.content, Triple)]

After second conversation:
ExtractedMemory(id='1d6c0292-a16c-468e-91c8-9ed06e742c36', content=Triple(subject='Bob', predicate='leads', object='ML team', context=None))
ExtractedMemory(id='5ef43e44-ff1c-4475-8f6b-aaceeca4e37e', content=Triple(subject='Bob', predicate='leads', object='NLP project', context=None))
ExtractedMemory(id='cd15b8f4-29ea-44bb-be1f-c11adaa8ae2f', content=Triple(subject='Alice', predicate='mentors', object='Bob', context=None))
ExtractedMemory(id='3dbecb1f-fde0-41d9-95ca-914eb639cc1a', content=Triple(subject='Bob', predicate='is on', object='ML team', context=None))


In [9]:
# Delete triples about an entity
conversation3 = [
    {"role": "user", "content": "Alice left the company."}
]
final = manager.invoke({"messages": conversation3, "existing": existing})
print("After third conversation:")
for m in final:
    print(m)
# ExtractedMemory(id='7ca76217-66a4-4041-ba3d-46a03ea58c1b', content=RemoveDoc(json_doc_id='f1bf258c-281b-4fda-b949-0c1930344d59'))
# ExtractedMemory(id='35b443c7-49e2-4007-8624-f1d6bcb6dc69', content=RemoveDoc(json_doc_id='0214f151-b0c5-40c4-b621-db36b845956c'))
# ExtractedMemory(id='65fd9b68-77a7-4ea7-ae55-66e1dd603046', content=RemoveDoc(json_doc_id='f1bf258c-281b-4fda-b949-0c1930344d59'))
# ExtractedMemory(id='7f8be100-5687-4410-b82a-fa1cc8d304c0', content=Triple(subject='Bob', predicate='leads', object='ML_team', context=None))
# ExtractedMemory(id='f4c09154-2557-4e68-8145-8ccd8afd6798', content=Triple(subject='Bob', predicate='leads', object='NLP_project', context=None))
# ExtractedMemory(id='f1bf258c-281b-4fda-b949-0c1930344d59', content=Triple(subject='Alice', predicate='manages', object='ML_team', context=None))
# ExtractedMemory(id='0214f151-b0c5-40c4-b621-db36b845956c', content=Triple(subject='Alice', predicate='mentors', object='Bob', context=None))
# ExtractedMemory(id='258dbf2d-e4ac-47ac-8ffe-35c70a3fe7fc', content=Triple(subject='Bob', predicate='is_member_of', object='ML_team', context=None))

After third conversation:
ExtractedMemory(id='cd15b8f4-29ea-44bb-be1f-c11adaa8ae2f', content=RemoveDoc(json_doc_id='cd15b8f4-29ea-44bb-be1f-c11adaa8ae2f'))
ExtractedMemory(id='1d6c0292-a16c-468e-91c8-9ed06e742c36', content=Triple(subject='Bob', predicate='leads', object='ML team', context=None))
ExtractedMemory(id='5ef43e44-ff1c-4475-8f6b-aaceeca4e37e', content=Triple(subject='Bob', predicate='leads', object='NLP project', context=None))
ExtractedMemory(id='3dbecb1f-fde0-41d9-95ca-914eb639cc1a', content=Triple(subject='Bob', predicate='is on', object='ML team', context=None))


In [10]:
from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint
from langgraph.store.memory import InMemoryStore
from langmem import create_memory_store_manager
from pydantic import BaseModel, Field

class Triple(BaseModel): 
    """Store all new facts, preferences, and relationships as triples."""
    subject: str
    predicate: str
    object: str
    context: str | None = None
    
# Set up store and models
store = InMemoryStore(  
    index={
        "dims": 768,
        "embed": emb_model,
    }
)
manager = create_memory_store_manager(
    model,
    namespace=("chat", "{user_id}", "triples"),  
    schemas=[Triple],
    instructions="Extract all user information and events as triples.",
    enable_inserts=True,
    enable_deletes=True,
)
my_llm = init_chat_model(model)

In [11]:
# Define app with store context
@entrypoint(store=store) 
def app(messages: list):
    response = my_llm.invoke(
        [
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            *messages
        ]
    )

    # Extract and store triples (Uses store from @entrypoint context)
    manager.invoke({"messages": messages}) 
    return response

In [12]:
# First conversation
app.invoke(
    [
        {
            "role": "user",
            "content": "Alice manages the ML team and mentors Bob, who is also on the team.",
        },
    ],
    config={"configurable": {"user_id": "user123"}},
)

# Second conversation
app.invoke(
    [
        {"role": "user", "content": "Bob now leads the ML team and the NLP project."},
    ],
    config={"configurable": {"user_id": "user123"}},
)

# Third conversation
app.invoke(
    [
        {"role": "user", "content": "Alice left the company."},
    ],
    config={"configurable": {"user_id": "user123"}},
)

# Check stored triples
for item in store.search(("chat", "user123")):
    print(item.namespace, item.value)

# Output:
# ('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Bob', 'predicate': 'is_member_of', 'object': 'ML_team', 'context': None}}
# ('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Bob', 'predicate': 'leads', 'object': 'ML_team', 'context': None}}
# ('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Bob', 'predicate': 'leads', 'object': 'NLP_project', 'context': None}}
# ('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Alice', 'predicate': 'employment_status', 'object': 'left_company', 'context': None}}

('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Bob', 'predicate': 'is on team', 'object': 'ML team', 'context': 'session_07c37c15-7481-47cb-9707-e78e36bdd11f'}}
('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Bob', 'predicate': 'leads', 'object': 'NLP project', 'context': 'session_cb8dcef4-0786-4dea-9d1e-3af3042fdb07'}}
('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Bob', 'predicate': 'leads', 'object': 'ML team', 'context': 'session_cb8dcef4-0786-4dea-9d1e-3af3042fdb07'}}
('chat', 'user123', 'triples') {'kind': 'Triple', 'content': {'subject': 'Alice', 'predicate': 'left', 'object': 'company', 'context': 'session_13c93974-57c8-438d-aaef-600655836627'}}


## Episodic Memory

In [13]:
from pydantic import BaseModel, Field
from langmem import create_memory_manager

class Episode(BaseModel):
    """An episode captures how to handle a specific situation, including the reasoning process
    and what made it successful."""

    observation: str = Field(
        ..., 
        description="The situation and relevant context"
    )
    thoughts: str = Field(
        ...,
        description="Key considerations and reasoning process"
    )
    action: str = Field(
        ...,
        description="What was done in response"
    )
    result: str = Field(
        ...,
        description="What happened and why it worked"
    )

manager = create_memory_manager(
    model,
    schemas=[Episode],
    instructions="Extract examples of successful interactions. Include the context, thought process, and why the approach worked.",
    enable_inserts=True,
)

# Example conversation
conversation = [
    {"role": "user", "content": "What's a binary tree? I work with family trees if that helps"},
    {"role": "assistant", "content": "A binary tree is like a family tree, but each parent has at most 2 children. Here's a simple example:\n   Bob\n  /  \\\nAmy  Carl\n\nJust like in family trees, we call Bob the 'parent' and Amy and Carl the 'children'."},
    {"role": "user", "content": "Oh that makes sense! So in a binary search tree, would it be like organizing a family by age?"},
]

In [14]:
# Extract episode(s)
episodes = manager.invoke({"messages": conversation})
# Example episode:
# [
#     ExtractedMemory(
#         id="f9194af3-a63f-4d8a-98e9-16c66e649844",
#         content=Episode(
#             observation="User struggled debugging a recursive "
#                         "function for longest path in binary "
#                         "tree, unclear on logic.",
#             thoughts="Used explorer in treehouse village "
#                      "metaphor to explain recursion:\n"
#                      "- Houses = Nodes\n"
#                      "- Bridges = Edges\n"
#                      "- Explorer's path = Traversal",
#             action="Reframed problem using metaphor, "
#                    "outlined steps:\n"
#                    "1. Check left path\n"
#                    "2. Check right path\n"
#                    "3. Add 1 for current position\n"
#                    "Highlighted common bugs",
#             result="Metaphor helped user understand logic. "
#                    "Worked because it:\n"
#                    "1. Made concepts tangible\n"
#                    "2. Created mental model\n"
#                    "3. Showed key steps\n"
#                    "4. Pointed to likely bugs",
#         ),
#     )
# ]

In [15]:
episodes

[ExtractedMemory(id='ffb883ca-0cc3-4476-b10e-9bc6cf055b81', content=Episode(observation='The user is asking to understand what a binary tree is, and then a binary search tree, and has a background in family trees. They are looking for an explanation that connects to their existing knowledge.', thoughts="Leverage the user's familiarity with family trees to explain the concept of a binary tree, and then extend that analogy to explain a binary search tree. This approach makes complex technical concepts more accessible and relatable.", action='Explained a binary tree by comparing it to a family tree where each parent has at most two children, providing a simple visual example. The AI then prompted the user to extend this analogy to a binary search tree.', result="The user understood the initial explanation of a binary tree and was able to build upon the analogy to inquire about binary search trees, indicating the effectiveness of the relatable analogy in conveying complex information. The 

## Procedural Memory - System Prompt Optimization

In [16]:
from langmem import create_prompt_optimizer

optimizer = create_prompt_optimizer(
    model,
    kind="metaprompt",
    config={"max_reflection_steps": 3}
)

In [17]:
prompt = "You are a helpful assistant."
trajectory = [
    {"role": "user", "content": "Explain inheritance in Python"},
    {"role": "assistant", "content": "Here's a detailed theoretical explanation..."},
    {"role": "user", "content": "Show me a practical example instead"},
]
optimized = optimizer.invoke({
    "trajectories": [(trajectory, {"user_score": 0})], 
    "prompt": prompt
})
print(optimized)

You are a helpful assistant. When explaining technical concepts, prioritize practical examples.
